In [1]:
# 1 Count Even Numbers

In [2]:
!nvidia-smi

Mon Jun  1 05:43:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install numba

In [6]:
import numpy as np
import time
from numba import cuda

# Dataset
N = 10_000_000
data = np.random.randint(0, 1000000, N).astype(np.int32)

# Result array
result = np.zeros(1, dtype=np.int32)

# CUDA Kernel
@cuda.jit
def count_even_kernel(arr, result):

    idx = cuda.grid(1)

    if idx < arr.size:

        if arr[idx] % 2 == 0:

            cuda.atomic.add(result, 0, 1)

# Copy to GPU
d_arr = cuda.to_device(data)
d_result = cuda.to_device(result)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

start = time.time()

count_even_kernel[blocks_per_grid, threads_per_block](
    d_arr,
    d_result
)

cuda.synchronize()

gpu_time = time.time() - start

answer = d_result.copy_to_host()

print("Even Numbers =", answer[0])
print("GPU Time =", gpu_time, "seconds")

Even Numbers = 4999115
GPU Time = 1.6184008121490479 seconds
